In [12]:
import pandas as pd
import numpy as np

In [13]:
train = pd.read_csv('../data/processed/train_raw.csv')
oil = pd.read_csv('../data/raw/oil.csv')
holidays = pd.read_csv('../data/raw/holidays_events.csv')
stores = pd.read_csv('../data/raw/stores.csv')
transactions = pd.read_csv('../data/raw/transactions.csv')

In [14]:
train['date'] = pd.to_datetime(train['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])
transactions['date'] = pd.to_datetime(transactions['date'])

print('Date ranges:')
print(f'train: {train["date"].min()} to {train["date"].max()}')
print(f'oil: {oil["date"].min()} to {oil["date"].max()}')
print(f'transactions: {transactions["date"].min()} to {transactions["date"].max()}')

Date ranges:
train: 2013-01-01 00:00:00 to 2017-08-15 00:00:00
oil: 2013-01-01 00:00:00 to 2017-08-31 00:00:00
transactions: 2013-01-01 00:00:00 to 2017-08-15 00:00:00


In [15]:
oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(method='linear')

print('Oil nulls after interpolation:', oil['dcoilwtico'].isnull().sum())

oil['dcoilwtico'] = oil['dcoilwtico'].bfill()

print('Oil nulls after fill:', oil['dcoilwtico'].isnull().sum())

Oil nulls after interpolation: 1
Oil nulls after fill: 0


In [16]:
national_holidays = holidays[
    (holidays['type'] == 'Holiday') & 
    (holidays['locale'] == 'National')
][['date', 'description']].copy()

national_holidays['is_holiday'] = 1

print(national_holidays.shape)
national_holidays.head()

(60, 3)


,date,description,is_holiday
14,2012-08-10,Primer Grito de Independencia,1
19,2012-10-09,Independencia de Guayaquil,1
21,2012-11-02,Dia de Difuntos,1
22,2012-11-03,Independencia de Cuenca,1
37,2012-12-25,Navidad,1


In [17]:
train = train.merge(oil, on='date', how='left')

train = train.merge(stores, on='store_nbr', how='left')

train = train.merge(transactions, on=['date', 'store_nbr'], how='left')

train = train.merge(national_holidays[['date', 'is_holiday']], on='date', how='left')
train['is_holiday'] = train['is_holiday'].fillna(0).astype(int)

print(train.shape)
train.head()

(3000888, 13)


,id,date,store_nbr,family,sales,onpromotion,dcoilwtico,city,state,type,cluster,transactions,is_holiday
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,93.14,Quito,Pichincha,D,13,NaN,1
1,1,2013-01-01,1,BABY CARE,0.0,0,93.14,Quito,Pichincha,D,13,NaN,1
2,2,2013-01-01,1,BEAUTY,0.0,0,93.14,Quito,Pichincha,D,13,NaN,1
3,3,2013-01-01,1,BEVERAGES,0.0,0,93.14,Quito,Pichincha,D,13,NaN,1
4,4,2013-01-01,1,BOOKS,0.0,0,93.14,Quito,Pichincha,D,13,NaN,1


In [18]:
train = train.drop(columns=['id'])

In [19]:
train['transactions'] = train['transactions'].fillna(0)

print('Nulls after merge:')
print(train.isnull().sum()[train.isnull().sum() > 0])

Nulls after merge:
dcoilwtico    857142
dtype: int64


In [20]:
train['dcoilwtico'] = train['dcoilwtico'].interpolate(method='linear').bfill()

print('Nulls after fill:', train['dcoilwtico'].isnull().sum())

Nulls after fill: 0


In [21]:
train.to_csv('../data/processed/train_cleaned.csv', index=False)